# SEER


## 1 · Environment setup


In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Point this at the folder containing config.py, run_experiment.py, etc.
    CODE_DIR = '/content/drive/MyDrive/SEER/code'
    os.chdir(CODE_DIR)

print('Working dir:', os.getcwd())


## 2 · Install dependencies


In [ ]:
%pip install -q -r requirements.txt

import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')


## 3 · Configure paths

`BIRD_ROOT` must point at an unzipped [BIRD-SQL](https://bird-bench.github.io/) dev release containing `dev/dev.json` and `dev/dev_databases/`. Set `HF_TOKEN` in the environment first if the models require authentication.


In [ ]:
os.environ.setdefault('BIRD_ROOT', './bird')
os.environ.setdefault('SEER_RESULTS', './results')
os.environ.setdefault('HF_HOME', './hfcache')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

Path(os.environ['SEER_RESULTS']).mkdir(parents=True, exist_ok=True)

print('BIRD_ROOT    =', os.environ['BIRD_ROOT'])
print('SEER_RESULTS =', os.environ['SEER_RESULTS'])
print('HF_HOME      =', os.environ['HF_HOME'])


## 4 · Verify the dataset loads


In [ ]:
import config, bird_loader

print(f'MAX_SAMPLES={config.MAX_SAMPLES}  SEED={config.SHUFFLE_SEED}  K_MAX={config.K_MAX}  '
      f'GATE_REDUCTION={config.GATE_REDUCTION}  EXEC_GUIDED={config.EXEC_GUIDED}  '
      f'LOAD_IN_4BIT={config.LOAD_IN_4BIT}')

qs = bird_loader.load_questions(config.SPLIT, limit=config.MAX_SAMPLES)
q = qs[0]
schema = bird_loader.get_schema(q['db_id'], config.SPLIT)
print(f"Loaded {len(qs)} questions | first db_id: {q['db_id']} | schema {len(schema)} chars")
print('Dataset OK — ready to run.')


## 5 · Calibration thresholds

τ = μ + 2σ per (model, gate), computed on the synthetic calibration set and cached in `results/calibration.json`.


In [ ]:
import json

calib_path = Path(os.environ['SEER_RESULTS']) / 'calibration.json'
calib = json.load(open(calib_path))

def _fmt(v):
    return f'{v:10.4f}' if isinstance(v, (int, float)) else f'{"n/a":>10s}'

print(f"{'model':10s} {'gate':8s} {'n':>4s} {'mu':>10s} {'sigma':>10s} {'tau':>10s}")
print('-' * 58)
for key, rec in calib.items():
    model_key, gate = key.split('/')
    print(f"{model_key:10s} {rec['gate']:8s} {rec['n_calibration']:>4d} "
          f"{_fmt(rec['mu'])} {_fmt(rec['sigma'])} {_fmt(rec['tau'])}")


## 6 · Standard baseline


In [ ]:
!python -u run_experiment.py --standard-only


## 7 · SEER — entropy and energy gates


In [ ]:
!python run_experiment.py --seer-only


## 8 · Tables & detector analysis

`analyze.py` → Table I, Table II, risk–coverage.

`analyze_detector.py` → first-token vs sequence-level signal comparison.


In [ ]:
!python analyze.py


In [ ]:
!python analyze_detector.py


## 9 · Inspect and archive results


In [ ]:
import json
import shutil

rd = Path(os.environ['SEER_RESULTS'])
print('--- SUMMARIES ---')
for f in sorted(rd.glob('*/summary_*.json')):
    d = json.load(open(f))
    print(f"{f.parent.name:10s} {d.get('config','?'):14s} "
          f"EX={100*d.get('execution_accuracy',0):.1f}%  "
          f"valid={100*d.get('valid_sql_rate',0):.1f}%  "
          f"steps={d.get('avg_steps',0):.2f}  conv={100*d.get('convergence_rate',0):.1f}%")

archive_path = Path('SEER_results')
shutil.make_archive(str(archive_path), 'zip', rd)
print(f'Zipped to {archive_path}.zip')

if IN_COLAB:
    from google.colab import files
    files.download(f'{archive_path}.zip')
